# 直方图与正态分位图完整教程：从图形绘制到分布判断

## 📚 教学目标
1. 理解直方图与频数分布表的关系
2. 掌握直方图的绘制方法和参数选择（组数、组距）
3. 学会用直方图判断数据分布形态（对称、右偏、左偏）
4. 理解正态分位图（QQ plot）的原理和解读
5. 用 scipy.stats.probplot 绘制 QQ 图并判断正态性

**参考书**：《基础统计学(第14版)》（Triola）第 2-2 节
**教学策略**：先用极小数据集让你"看见"直方图的构建过程，再用 QQ 图做严格的正态性判断

---

## 1. 场景设定：直方图是什么？

### 🎯 一个直觉问题

在上一节中，我们用频数分布表整理了数据。但表格不够直观——**直方图就是频数分布表的图形化版本**。

### 📖 书中的定义

> 直方图是由彼此相邻（除非数据中有间隔）且宽度相等的长条所组成的图表，其横坐标代表定量数据的不同分组，纵坐标为频数。长条的高度即为对应的频数。

### 💡 直方图的重要应用场景

- 数据分布形状的可视化
- 显示数据中心的位置
- 显示数据的分布情况
- 检测异常值

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import matplotlib.font_manager as fm
_cn_fonts = [f.name for f in fm.fontManager.ttflist if any(kw in f.name for kw in ['Hei', 'Song', 'PingFang', 'Arial Unicode', 'Noto Sans CJK', 'SimHei', 'Microsoft YaHei'])]
plt.rcParams['font.sans-serif'] = _cn_fonts[:3] + ['DejaVu Sans'] if _cn_fonts else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 从 20 个数据点开始：手绘直方图

### 🎯 场景

使用上一节的 20 个学生考试成绩，手动构建直方图。

In [ ]:
# ========== 构造微型数据集 ==========
scores = [55, 62, 68, 72, 75, 78, 80, 82, 83, 85,
          86, 88, 89, 90, 91, 92, 93, 95, 97, 99]

print("📊 原始数据（20 个学生考试成绩）:")
print(f"  {scores}")
print(f"  数据量: n = {len(scores)}")

In [ ]:
# ========== 绘制直方图 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：频数直方图
ax1 = axes[0]
bins = [49, 59, 69, 79, 89, 99]
ax1.hist(scores, bins=bins, color='steelblue', alpha=0.7, edgecolor='white', rwidth=0.95)
ax1.set_xlabel('Score', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Frequency Histogram (n=20)', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# 右图：相对频数直方图
ax2 = axes[1]
ax2.hist(scores, bins=bins, density=True, color='#e74c3c', alpha=0.7, edgecolor='white', rwidth=0.95)
ax2.set_xlabel('Score', fontsize=12)
ax2.set_ylabel('Relative Frequency (Density)', fontsize=12)
ax2.set_title('Relative Frequency Histogram (n=20)', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明:")
print(f"  左图：频数直方图，纵轴是每组的数据个数")
print(f"  右图：相对频数直方图，纵轴是密度（面积之和=1）")
print(f"  两个图的形状完全相同，只是纵轴刻度不同")

---

## 3. 直方图的参数选择

### 📐 组数的选择

组数不同，直方图的形状会有很大差异。我们来看一个例子。

| 组数 | 特点 |
|------|------|
| 太少（如 3 组） | 掩盖数据的细节特征 |
| 适中（5~20 组） | 能看出分布规律 |
| 太多（如 50 组） | 每组频数太少，噪声大 |

In [ ]:
# ========== 不同组数的直方图对比 ==========
data = np.random.normal(loc=70, scale=10, size=200)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

bin_counts = [5, 15, 40]
colors = ['#2ecc71', 'steelblue', '#e74c3c']

for ax, n_bins, color in zip(axes, bin_counts, colors):
    ax.hist(data, bins=n_bins, color=color, alpha=0.7, edgecolor='white')
    ax.set_xlabel('Score', fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    ax.set_title(f'Histogram with {n_bins} bins', fontsize=14, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明:")
print(f"  左图（5 组）：过于粗糙，看不清分布细节")
print(f"  中图（15 组）：适中，能清楚看到钟形分布")
print(f"  右图（40 组）：过于细碎，噪声太大")
print(f"  推荐：数据量 n 在 50~200 时，10~20 组比较合适")

---

## 4. 用直方图判断分布形态

### 📖 书中的四种常见分布形状

1. **钟形分布（正态分布）**：中间高，两边低，左右对称
2. **均匀分布**：各组频数大致相等
3. **右偏态分布**：高峰在左侧，尾部向右延伸
4. **左偏态分布**：高峰在右侧，尾部向左延伸

In [ ]:
# ========== 四种分布形态的直方图 ==========

# 生成四种分布的数据
normal_data = np.random.normal(loc=50, scale=10, size=500)
uniform_data = np.random.uniform(low=20, high=80, size=500)
right_skew = np.random.lognormal(mean=3.5, sigma=0.4, size=500)
left_skew = 100 - np.random.lognormal(mean=3.5, sigma=0.4, size=500)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

datasets = [normal_data, uniform_data, right_skew, left_skew]
titles = ['Normal (Bell-shaped)', 'Uniform', 'Right-skewed (Positive)', 'Left-skewed (Negative)']
colors = ['steelblue', '#2ecc71', '#e74c3c', '#e67e22']

for ax, data, title, color in zip(axes.flat, datasets, titles, colors):
    ax.hist(data, bins=20, color=color, alpha=0.7, edgecolor='white', density=True)
    ax.set_xlabel('Value', fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明:")
print(f"  左上：钟形分布 - 中间高，两边低，左右对称")
print(f"  右上：均匀分布 - 各组频数大致相等")
print(f"  左下：右偏态 - 高峰在左侧，尾部向右延伸（如收入数据）")
print(f"  右下：左偏态 - 高峰在右侧，尾部向左延伸（如考试成绩）")

---

## 5. 扩展到 500 个数据点：真实规模的直方图

### 🎯 场景

模拟 500 个通勤时间数据，绘制直方图并判断分布形态。

### 📐 数据生成过程 (DGP)

- 通勤时间服从右偏分布
- 使用对数正态分布模拟

In [ ]:
# ========== 数据生成过程 (DGP) ==========
n = 500
commute_time = np.random.lognormal(mean=3.2, sigma=0.5, size=n).astype(int)
commute_time = np.clip(commute_time, 5, 150)

print(f"📊 模拟参数:")
print(f"  样本量: n = {n}")
print(f"  分布类型: 对数正态分布 LogNormal(μ=3.2, σ=0.5)")
print(f"  均值: {commute_time.mean():.1f} 分钟")
print(f"  中位数: {np.median(commute_time):.1f} 分钟")
print(f"  标准差: {commute_time.std():.1f} 分钟")

In [ ]:
# ========== 绘制 500 个数据点的直方图 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：频数直方图
ax1 = axes[0]
ax1.hist(commute_time, bins=15, color='steelblue', alpha=0.7, edgecolor='white')
ax1.axvline(commute_time.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean = {commute_time.mean():.1f}')
ax1.axvline(np.median(commute_time), color='green', linestyle='--', linewidth=2, label=f'Median = {np.median(commute_time):.1f}')
ax1.set_xlabel('Commute Time (min)', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Frequency Histogram (n=500)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# 右图：带正态曲线的直方图
ax2 = axes[1]
ax2.hist(commute_time, bins=15, density=True, color='steelblue', alpha=0.7, edgecolor='white')
x = np.linspace(commute_time.min(), commute_time.max(), 100)
y = stats.norm.pdf(x, commute_time.mean(), commute_time.std())
ax2.plot(x, y, 'r-', linewidth=2, label='Normal Curve')
ax2.set_xlabel('Commute Time (min)', fontsize=12)
ax2.set_ylabel('Density', fontsize=12)
ax2.set_title('Histogram with Normal Curve Overlay', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明:")
print(f"  左图：红色虚线是均值，绿色虚线是中位数")
print(f"  右图：红色曲线是理论正态分布，与实际数据不完全吻合")
print(f"  均值 > 中位数，说明数据右偏（正偏态）")

---

## 6. 正态分位图（QQ 图）

### 📐 QQ 图的原理

正态分位图（QQ plot）是一种更客观的方法来判断数据是否服从正态分布。

**原理**：
- 横轴：理论分位数（如果数据服从正态分布，应该有的值）
- 纵轴：实际数据的分位数
- 如果数据服从正态分布，点应该落在一条直线上

### 📖 书中的判断标准

> **正态分布**：如果正态分位图中点的分布相当接近直线，并且这些点没有任何非直线的系统性特征，那么该总体呈正态分布。
>
> **非正态分布**：如果正态分位图满足以下两个条件中的一个或两个，那么该总体不呈正态分布。
> - 这些点完全不接近直线
> - 这些点显示出某种非直线的系统性特征

In [ ]:
# ========== 绘制 QQ 图 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：正态数据的 QQ 图
normal_data = np.random.normal(loc=70, scale=10, size=200)
stats.probplot(normal_data, dist="norm", plot=axes[0])
axes[0].set_title('QQ Plot: Normal Data', fontsize=14, fontweight='bold')
axes[0].get_lines()[0].set_markerfacecolor('steelblue')
axes[0].get_lines()[0].set_markeredgecolor('steelblue')
axes[0].get_lines()[0].set_markersize(6)
axes[0].get_lines()[1].set_color('red')

# 右图：右偏数据的 QQ 图
stats.probplot(commute_time, dist="norm", plot=axes[1])
axes[1].set_title('QQ Plot: Right-skewed Data', fontsize=14, fontweight='bold')
axes[1].get_lines()[0].set_markerfacecolor('#e74c3c')
axes[1].get_lines()[0].set_markeredgecolor('#e74c3c')
axes[1].get_lines()[0].set_markersize(6)
axes[1].get_lines()[1].set_color('red')

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明:")
print(f"  左图：正态数据的 QQ 图 - 点紧密围绕在红线周围")
print(f"  右图：右偏数据的 QQ 图 - 点在两端偏离红线")
print(f"  右偏数据的特征：右端向上弯曲，左端向下弯曲")

In [ ]:
# ========== 四种分布的 QQ 图对比 ==========

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

datasets = [
    np.random.normal(loc=50, scale=10, size=200),
    np.random.uniform(low=20, high=80, size=200),
    np.random.lognormal(mean=3.5, sigma=0.4, size=200),
    100 - np.random.lognormal(mean=3.5, sigma=0.4, size=200)
]
titles = ['Normal', 'Uniform', 'Right-skewed', 'Left-skewed']

for ax, data, title in zip(axes.flat, datasets, titles):
    stats.probplot(data, dist="norm", plot=ax)
    ax.set_title(f'QQ Plot: {title}', fontsize=13, fontweight='bold')
    ax.get_lines()[0].set_markersize(5)
    ax.get_lines()[1].set_color('red')

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明:")
print(f"  左上：正态分布 - 点紧密围绕红线")
print(f"  右上：均匀分布 - 两端偏离，呈 S 形")
print(f"  左下：右偏分布 - 右端向上弯曲")
print(f"  右下：左偏分布 - 左端向下弯曲")

---

## 7. 用 scipy 进行正态性检验

### 🔬 Shapiro-Wilk 检验

除了看 QQ 图，还可以用 Shapiro-Wilk 检验来判断数据是否服从正态分布。

- **H₀**: 数据服从正态分布
- **H₁**: 数据不服从正态分布
- **α = 0.05**

In [ ]:
# ========== Shapiro-Wilk 正态性检验 ==========
print("🔬 Shapiro-Wilk 正态性检验")
print("─" * 50)

test_data = [
    ('Normal', np.random.normal(loc=50, scale=10, size=200)),
    ('Uniform', np.random.uniform(low=20, high=80, size=200)),
    ('Right-skewed', np.random.lognormal(mean=3.5, sigma=0.4, size=200)),
    ('Commute Time', commute_time)
]

for name, data in test_data:
    stat, p_value = stats.shapiro(data)
    is_normal = "✓ 正态" if p_value > 0.05 else "✗ 非正态"
    print(f"\n  {name}:")
    print(f"    W = {stat:.6f}, p = {p_value:.6f} → {is_normal}")

print(f"\n💡 解读:")
print(f"  - p > 0.05：不拒绝正态分布假设（可能是正态）")
print(f"  - p ≤ 0.05：拒绝正态分布假设（不是正态）")

---

## 8. 综合案例：通勤时间的完整分析

### 🎯 目标

综合使用直方图和 QQ 图，判断通勤时间是否服从正态分布。

In [ ]:
# ========== 综合分析 ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 图1：直方图
ax1 = axes[0]
ax1.hist(commute_time, bins=15, density=True, color='steelblue', alpha=0.7, edgecolor='white')
x = np.linspace(commute_time.min(), commute_time.max(), 100)
y = stats.norm.pdf(x, commute_time.mean(), commute_time.std())
ax1.plot(x, y, 'r-', linewidth=2, label='Normal Curve')
ax1.set_xlabel('Commute Time (min)', fontsize=12)
ax1.set_ylabel('Density', fontsize=12)
ax1.set_title('Histogram with Normal Curve', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# 图2：QQ 图
ax2 = axes[1]
stats.probplot(commute_time, dist="norm", plot=ax2)
ax2.set_title('QQ Plot', fontsize=14, fontweight='bold')
ax2.get_lines()[0].set_markerfacecolor('steelblue')
ax2.get_lines()[0].set_markeredgecolor('steelblue')
ax2.get_lines()[0].set_markersize(6)
ax2.get_lines()[1].set_color('red')

# 图3：箱形图
ax3 = axes[2]
bp = ax3.boxplot(commute_time, vert=True, patch_artist=True)
bp['boxes'][0].set_facecolor('steelblue')
bp['boxes'][0].set_alpha(0.7)
ax3.set_ylabel('Commute Time (min)', fontsize=12)
ax3.set_title('Boxplot', fontsize=14, fontweight='bold')
ax3.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# 统计检验
stat, p_value = stats.shapiro(commute_time)
skewness = stats.skew(commute_time)
kurtosis = stats.kurtosis(commute_time)

print(f"\n📊 综合分析结果:")
print(f"  Shapiro-Wilk 检验: W = {stat:.6f}, p = {p_value:.6f}")
print(f"  偏度: {skewness:.4f} (> 0 表示右偏)")
print(f"  峰度: {kurtosis:.4f}")

if p_value < 0.05:
    print(f"\n  💡 结论: 数据不服从正态分布（p < 0.05）")
    print(f"  QQ 图显示右端向上弯曲，符合右偏分布的特征")
else:
    print(f"\n  💡 结论: 数据可能服从正态分布（p ≥ 0.05）")

---

## 📌 核心概念回顾

### 📌 直方图 (Histogram)
- **定义**: 由宽度相等的长条组成的图表，横轴是数据分组，纵轴是频数
- **作用**: 可视化数据分布形态、检测异常值
- **参数**: 组数（bins）的选择影响图形的细节程度

### 📌 分布形态 (Distribution Shape)
- **钟形分布**: 中间高，两边低，左右对称
- **均匀分布**: 各组频数大致相等
- **右偏态**: 高峰在左侧，尾部向右延伸（如收入）
- **左偏态**: 高峰在右侧，尾部向左延伸（如考试成绩）

### 📌 正态分位图 (QQ Plot)
- **原理**: 比较实际分位数与理论正态分位数
- **判断标准**: 点在直线上 → 正态；点偏离直线 → 非正态
- **右偏特征**: 右端向上弯曲
- **左偏特征**: 左端向下弯曲

### 📌 Shapiro-Wilk 检验
- **H₀**: 数据服从正态分布
- **判断**: p > 0.05 → 不拒绝正态；p ≤ 0.05 → 拒绝正态

### 🔗 完整流程
```
绘制直方图 → 观察分布形态 → 绘制 QQ 图 → Shapiro-Wilk 检验 → 综合判断
    ↓              ↓              ↓              ↓              ↓
  可视化       初步判断       图形验证       统计检验       最终结论
```

### 📝 正态性判断方法汇总

| 方法 | 类型 | 优点 | 缺点 |
|------|------|------|------|
| 直方图 | 图形法 | 直观 | 主观性强 |
| QQ 图 | 图形法 | 能看出偏离方向 | 需要经验 |
| Shapiro-Wilk | 统计检验 | 客观、精确 | 小样本敏感 |
| 偏度/峰度 | 描述统计 | 简单 | 不能直接判断 |

---

## ❌ 常见误区

### ❌ 误区 1: 直方图看起来像钟形就是正态分布
**✓ 正确理解**: 直方图只是初步判断，组数不同会导致形状差异。需要结合 QQ 图和 Shapiro-Wilk 检验综合判断。

### ❌ 误区 2: QQ 图上的点必须完美落在直线上才算正态
**✓ 正确理解**: 实际数据总是有随机波动的。只要点大致围绕直线，没有明显的系统性偏离，就可以认为是正态的。

### ❌ 误区 3: Shapiro-Wilk 检验 p < 0.05 就一定不是正态分布
**✓ 正确理解**: 样本量很大时，即使轻微偏离正态也会导致 p < 0.05。需要结合实际意义判断。

### ❌ 误区 4: 偏度 > 0 就是右偏，偏度 < 0 就是左偏
**✓ 正确理解**: 偏度的绝对值需要足够大（如 > 0.5）才能认为有实际意义的偏态。接近 0 时可能是近似对称的。

### ❌ 误区 5: 正态性检验不重要，看直方图就够了
**✓ 正确理解**: 后续的统计推断（如 t 检验、方差分析）通常要求数据来自正态总体。正态性检验是必要的步骤。